# Q-Former A/B — cải tiến projector (từ `giai_phap_tinh_chinh_kien_truc.html`)

**Giả thuyết:** projector `Linear(192→1 token)` là cổ chai — nén toàn bộ ảnh về 1 token cho GPT-2.
**Thử:** thay bằng **Q-Former nhẹ** (K=16 query + 2 lớp cross-attention lên patch token CLIP) → K+1 prefix token.

### Thiết kế A/B (train TỪ ĐẦU, cùng recipe, chỉ khác projector)
| Run | image_proj | Kiến trúc |
|---|---|---|
| `Q_pooled` | pooled | baseline (1 prefix token) |
| `Q_qformer` | qformer | Q-Former nhẹ (17 prefix token) |

Cả hai: pubmedclip + pubmedbert + gpt2, **cùng** lr=1e-4, wd=1e-2, epochs/patience. Đo **bảng metric đầy đủ** (em_overall/closed/open + token_f1_open + sem_open) trên VQA-RAD + RadImageNet.

> Cần code mới (`--image_proj`, `src/models/qformer.py`) — cell 1.1 tự `git reset` lấy về. Pubmed runs chạy **trong kernel** (tránh treo BERT).

## BƯỚC 1: Setup

In [ ]:
# 1.1. Cap nhat code moi nhat (DAN token GitHub vao GITHUB_TOKEN, khong commit token)
GITHUB_TOKEN = ''   # <-- DAN TOKEN O DAY
GITHUB_USER  = 'HoangVuSnape'
REPO = '02_Code_DeepLearning'
import os
CLONE_URL = 'https://' + GITHUB_USER + ':' + GITHUB_TOKEN + '@github.com/' + GITHUB_USER + '/' + REPO + '.git'
if os.path.isdir(REPO + '/.git'):
    print('Repo ton tai -> git reset --hard origin/main...')
    get_ipython().system('cd ' + REPO + ' && git fetch origin && git reset --hard origin/main && git log --oneline -1')
elif not os.path.isdir('.git'):
    get_ipython().system('git clone ' + CLONE_URL)

In [ ]:
import os
if os.path.isdir('02_Code_DeepLearning'):
    os.chdir('02_Code_DeepLearning')
print('CWD:', os.getcwd())

In [ ]:
get_ipython().system('pip install -q datasets tqdm huggingface_hub comet_ml python-dotenv scikit-learn pytest hf-transfer transformers hf_xet sentence-transformers')
import os
from dotenv import load_dotenv
load_dotenv('.env') if os.path.exists('.env') else load_dotenv()
def _mask(t):
    return (t[:8] + '...' + t[-6:]) if t and len(t) > 16 else ('co' if t else 'TRONG')
for k in ['COMET_API_KEY', 'DISCORD_WEBHOOK_URL', 'HF_TOKEN']:
    print(' ', k, ':', _mask(os.environ.get(k, '')))

## BƯỚC 2: CONFIG

In [ ]:
# ================== CONFIG (Q-Former A/B) ==================
EPOCHS   = 100
PATIENCE = 15
LR_GPT2  = 1e-4     # recipe tuned cho GPT-2
WD_GPT2  = 1e-2
BATCH    = 32

COMET_PROJECT = 'medvqa-qformer-ab'
HF_REPO       = 'VQA-DeepLearning/vqa-rad-qformer-ab'   # repo RIENG cho thi nghiem nay

USE_HF_PUSH = True
USE_COMET   = True
USE_DISCORD = True
print('CONFIG:', EPOCHS, 'ep | pat', PATIENCE, '| lr', LR_GPT2, '| wd', WD_GPT2, '| repo', HF_REPO)

## BƯỚC 3: RUN MATRIX (2 run A/B) + HELPER

In [ ]:
# 2 run: chi khac 'proj'. Ca hai pubmedclip+pubmedbert+gpt2, cung recipe.
Q_pooled  = {'name': 'Q_pooled',  'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'proj': 'pooled'}
Q_qformer = {'name': 'Q_qformer', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'proj': 'qformer'}
QRUNS = [Q_pooled, Q_qformer]

def _model(r):
    f = ['--image_encoder', r['img'], '--text_encoder', r['txt'], '--decoder', r['dec']]
    if r.get('proj'): f += ['--image_proj', r['proj']]
    return f

def _common(push=True):
    f = ['--project_name', COMET_PROJECT]
    if USE_COMET:   f += ['--use_comet']
    if USE_DISCORD: f += ['--use_discord']
    if USE_HF_PUSH and push: f += ['--use_hf_push', '--hf_repo_id', HF_REPO]
    return f

def run_train_inkernel(r):
    # Pubmed -> chay TRONG KERNEL (tranh deadlock cold-start subprocess o BERT)
    a = ['--run_name', r['name']] + _model(r) + ['--batch_size', str(BATCH)]
    a += ['--epochs', str(EPOCHS), '--lr', str(LR_GPT2), '--weight_decay', str(WD_GPT2), '--patience', str(PATIENCE)]
    a += _common(push=True)
    print('%run train.py', ' '.join(a), flush=True)
    get_ipython().run_line_magic('run', 'train.py ' + ' '.join(a))

print('San sang. QRUNS =', [r['name'] for r in QRUNS])

## BƯỚC 4: Warm-cache (tránh 429/treo khi nạp model)
Tải trước model vào cache. KHÔNG bật offline (offline chặn push). Pubmed chạy trong kernel nên không treo.

In [ ]:
import os
for k in ['HF_HUB_OFFLINE', 'HF_DATASETS_OFFLINE', 'TRANSFORMERS_OFFLINE']:
    os.environ.pop(k, None)                      # KHONG offline -> push HF hoat dong
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
from transformers import CLIPVisionModel, AutoModel, AutoTokenizer, GPT2LMHeadModel
from datasets import load_dataset
_BERT = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
print('1/4 CLIP...', flush=True);       CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch32')
print('2/4 PubMedBERT...', flush=True); AutoModel.from_pretrained(_BERT); AutoTokenizer.from_pretrained(_BERT)
print('3/4 GPT2...', flush=True);       GPT2LMHeadModel.from_pretrained('gpt2')
print('4/4 VQA-RAD + MiniLM...', flush=True)
AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2'); AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
load_dataset('flaviagiammarino/vqa-rad')
print('=> CACHE XONG', flush=True)

## BƯỚC 5: Train A/B (trong kernel)

In [ ]:
run_train_inkernel(Q_pooled)     # baseline (1 prefix token)

In [ ]:
run_train_inkernel(Q_qformer)    # Q-Former nhe (K+1 prefix token)

## BƯỚC 6: EVAL ĐẦY ĐỦ (2-in-1) — cả 2 run trên VQA-RAD + RadImageNet
Truyền đúng `image_proj` khi build model (phải khớp lúc train).

In [ ]:
import os, json, time, io, torch
from PIL import Image
from datasets import load_dataset
from torch.utils.data import DataLoader
from src.models.fusion import build_model
from src.train.engine import predict_answers
from src.train.metrics import score_predictions
from src.data.dataset import VQARADClsDataset, default_transform, build_text_tokenizer
from src.data.vqa_rad import normalize_answer
from src.integrations import hf_push

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
PARQUET  = 'radimagenet_vqa_audit/radimagenet_vqa_500_test.parquet'
HF_TOKEN = os.environ.get('HF_TOKEN')

with open('runs/vocab_answer2id.json') as f: answer2id = json.load(f)
with open('runs/vocab_q_vocab.json')  as f: q_vocab   = json.load(f)
id2answer = {i: a for a, i in answer2id.items()}
yes_id, no_id = answer2id.get('yes'), answer2id.get('no')

print('Tai VQA-RAD test...')
vqarad = [dict(x) for x in load_dataset('flaviagiammarino/vqa-rad')['test']]
print('Tai RadImageNet parquet...')
rad = [dict(x) for x in load_dataset('parquet', data_files=PARQUET, split='train')]
for _r in rad:
    im = _r.get('image')
    if isinstance(im, dict) and im.get('bytes'):
        _r['image'] = Image.open(io.BytesIO(im['bytes']))
TESTS = {'vqarad': (vqarad, 'runs'), 'radimagenet': (rad, 'runs_radimagenet')}

def eval_both(r, push=True):
    name = r['name']
    path = 'runs/' + name + '_best.pt'
    if not os.path.exists(path):
        print('x khong co checkpoint:', path); return
    model = build_model(vocab_size=len(q_vocab), num_classes=len(answer2id),
                        image_encoder=r['img'], text_encoder=r['txt'], decoder=r['dec'],
                        pretrained=False, max_len=32, image_proj=r.get('proj', 'pooled'))
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device).eval()
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tok = build_text_tokenizer(r['txt'])
    tf  = default_transform(224, train=False, imagenet=False, clip=(r['img']=='pubmedclip'))
    print('>> [' + name + '] proj=' + r.get('proj','pooled') + ' loaded, eval 2 test...')
    for tag, (recs, outdir) in TESTS.items():
        loader = DataLoader(VQARADClsDataset(recs, answer2id, q_vocab, tf, max_len=32,
                                             text_tokenizer=tok), batch_size=32, shuffle=False)
        t0 = time.time()
        hyps, p_yes = predict_answers(model, loader, id2answer, q_vocab, device,
                                      yes_id=yes_id, no_id=no_id, constrained_closed=False)
        m = score_predictions([normalize_answer(x['answer']) for x in recs], hyps, p_yes=p_yes)
        m['params_trainable'] = n_train
        m['inference_time_sec'] = round(time.time() - t0, 3)
        os.makedirs(outdir, exist_ok=True)
        fp = outdir + '/' + name + '_metrics.json'
        with open(fp, 'w') as f: json.dump(m, f, indent=2)
        sub = name if tag == 'vqarad' else name + '/radimagenet'
        if push and HF_TOKEN:
            hf_push.push_files(HF_REPO, [(fp, sub + '/metrics.json')], HF_TOKEN)
        print('   ', tag, '| em', round(m['em_overall'], 4),
              '| f1_open', round(m.get('token_f1_open', 0), 4), '| sem_open', m.get('sem_open'))
    del model; torch.cuda.empty_cache()

for r in QRUNS:
    eval_both(r)
print('=== XONG EVAL A/B ===')

## BƯỚC 7: Bảng so sánh A/B

In [ ]:
# Bang so sanh pooled vs qformer tren VQA-RAD (in-domain)
import json
cols = ['em_overall','em_closed','em_open','token_f1_open','bleu1_open','sem_open','params_trainable']
print(f"{'run':<12} " + ' '.join(f'{c:<14}' for c in cols))
for r in QRUNS:
    fp = 'runs/' + r['name'] + '_metrics.json'
    try:
        m = json.load(open(fp))
        print(f"{r['name']:<12} " + ' '.join(f"{m.get(c, 0):<14}" for c in cols))
    except Exception as e:
        print(r['name'], 'chua co metrics:', e)
print('\n(delta = Q_qformer - Q_pooled: xem token_f1_open & sem_open co tang khong)')

In [ ]:
# Backup runs/ len HF
from huggingface_hub import HfApi
import os
try:
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(folder_path='runs', repo_id=HF_REPO, repo_type='model', allow_patterns=['*.json', '*.csv', '*.pt'])
    print('Backup runs/ xong')
except Exception as e:
    print('Backup bo qua:', e)

In [ ]:
# ②b: eval 1 checkpoint voi 3 kieu decode (greedy/beam/contrastive) - EVAL ONLY, khong train lai
import json, torch
from torch.utils.data import DataLoader

CKPT = Q_qformer   # doi thanh Q_pooled neu muon so tren baseline

def eval_decode(r, decode, push=True):
    name, path = r['name'], 'runs/' + r['name'] + '_best.pt'
    if not os.path.exists(path):
        print('x khong co checkpoint:', path); return None
    model = build_model(vocab_size=len(q_vocab), num_classes=len(answer2id),
                        image_encoder=r['img'], text_encoder=r['txt'], decoder=r['dec'],
                        pretrained=False, max_len=32, image_proj=r.get('proj', 'pooled'))
    model.load_state_dict(torch.load(path, map_location=device)); model.to(device).eval()
    tok = build_text_tokenizer(r['txt'])
    tf  = default_transform(224, train=False, imagenet=False, clip=(r['img'] == 'pubmedclip'))
    loader = DataLoader(VQARADClsDataset(vqarad, answer2id, q_vocab, tf, max_len=32, text_tokenizer=tok),
                        batch_size=32, shuffle=False)
    hyps, p_yes = predict_answers(model, loader, id2answer, q_vocab, device,
                                  yes_id=yes_id, no_id=no_id, constrained_closed=False, decode=decode)
    m = score_predictions([normalize_answer(x['answer']) for x in vqarad], hyps, p_yes=p_yes)
    fp = 'runs/' + name + '_decode_' + decode + '_metrics.json'
    with open(fp, 'w') as f:
        json.dump(m, f, indent=2)
    if push and HF_TOKEN:
        hf_push.push_files(HF_REPO, [(fp, name + '/decode_' + decode + '/metrics.json')], HF_TOKEN)
    del model
    torch.cuda.empty_cache()
    return m

print('CKPT =', CKPT['name'])
print('decode        em_overall   em_open   token_f1_open   sem_open')
for dec in ['greedy', 'beam', 'contrastive']:
    m = eval_decode(CKPT, dec)
    if m:
        print(dec.ljust(13),
              str(round(m['em_overall'], 4)).ljust(12),
              str(round(m['em_open'], 4)).ljust(9),
              str(round(m['token_f1_open'], 4)).ljust(15),
              m.get('sem_open'))

## ③ (Bonus) RL SCST với Semantic Reward
Fine-tune RL trên checkpoint SFT tốt (mặc định `Q_qformer`) với reward **semantic** (cosine sentence-embedding) thay cho lexical `0.5·EM+0.5·F1`. Đây là chỗ RL cũ (`D1_scst`) thất bại. So `D_semantic` với SFT gốc.

In [ ]:
# ③: RL SCST voi SEMANTIC reward, fine-tune tren checkpoint SFT (in-kernel)
BASE_SFT  = 'Q_qformer'    # checkpoint SFT lam diem xuat phat (doi Q_pooled neu muon)
RL_REWARD = 'semantic'     # 'semantic' | 'mixed' | 'lexical'
BASE_PROJ = 'qformer' if BASE_SFT == 'Q_qformer' else 'pooled'
D_sem = {'name': 'D_' + RL_REWARD, 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'proj': BASE_PROJ}

a = ['--run_name', D_sem['name']] + _model(D_sem) + ['--batch_size', str(BATCH)]
a += ['--rl', '--rl_epochs', '20', '--rl_lr', '1e-5', '--rl_reward', RL_REWARD,
      '--load_checkpoint', 'runs/' + BASE_SFT + '_best.pt']
a += _common(push=True)
print('%run train.py', ' '.join(a), flush=True)
get_ipython().run_line_magic('run', 'train.py ' + ' '.join(a))

# Eval D_sem (so voi SFT goc: xem em_open / token_f1_open / sem_open co tang khong)
eval_both(D_sem)